# Part 3 - Adversarial Attacks

Implement character-level evasion and label-flipping poisoning.

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from assignment_workflow import load_and_split_data, tokenize_df, perturb, train_distilbert, evaluate_binary

train_df, eval_df = load_and_split_data('jigsaw-unintended-bias-train.csv')
model_dir = 'saved_model/part1_baseline'
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
trainer = Trainer(model=model, args=TrainingArguments(output_dir='tmp_part3', report_to=[]))

In [ ]:
# Attack 1: Character-level evasion
eval_dataset = tokenize_df(eval_df, tokenizer)
clean_logits = trainer.predict(eval_dataset).predictions
clean_prob = torch.softmax(torch.tensor(clean_logits), dim=1).numpy()[:,1]
clean_pred = (clean_prob >= 0.5).astype(int)

candidates = eval_df[(clean_pred == 1) & (clean_prob >= 0.7)].sample(min(500, ((clean_pred == 1) & (clean_prob >= 0.7)).sum()), random_state=42)
perturbed_texts = candidates['comment_text'].fillna('').apply(perturb).tolist()

enc = tokenizer(perturbed_texts, truncation=True, padding=True, max_length=128, return_tensors='pt')
with torch.no_grad():
    out = model(**enc)
    pert_prob = torch.softmax(out.logits, dim=1).numpy()[:,1]
pert_pred = (pert_prob >= 0.5).astype(int)
asr = float((pert_pred == 0).mean())
print({'attack_success_rate': asr, 'avg_conf_before': float(clean_prob[candidates.index].mean()), 'avg_conf_after': float(pert_prob.mean())})

In [ ]:
# Attack 2: Label-flipping poisoning
poisoned = train_df.copy()
flip_idx = poisoned.sample(frac=0.05, random_state=42).index
poisoned.loc[flip_idx, 'label'] = 1 - poisoned.loc[flip_idx, 'label']

poison_trainer, poison_tokenizer = train_distilbert(poisoned, eval_df, output_dir='saved_model/part3_poisoned')
poison_eval_dataset = tokenize_df(eval_df, poison_tokenizer)
poison_prob = torch.softmax(torch.tensor(poison_trainer.predict(poison_eval_dataset).predictions), dim=1).numpy()[:,1]

baseline_metrics = evaluate_binary(eval_df['label'].values, clean_prob, threshold=0.5)
poison_metrics = evaluate_binary(eval_df['label'].values, poison_prob, threshold=0.5)
print('Baseline:', baseline_metrics)
print('Poisoned:', poison_metrics)